In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine
import os
print(os.getcwd())
import sys
print(sys.executable)

D:\Z\my-projects\data-science\HR-Analytics-Dashboard\sql
C:\ProgramData\anaconda3\python.exe


In [2]:
DB_CONFIG = {
    "user": "postgres",
    "password": "dunith",
    "host": "localhost",
    "port": "5432",
    "database": "hr_analytics",
}

CLEANED_CSV_PATH = r"D:\Z\my-projects\data-science\HR-Analytics-Dashboard\data\processed\hr_data_cleaned.csv"
TABLE_NAME = "hr_employees"

In [3]:
# Helper: convert Camelcase columns to snakecase
def camel_to_snake(name: str) -> str:
    name = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    return name.lower()

In [4]:
# LOAD CSV
df = pd.read_csv(CLEANED_CSV_PATH)   
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns from CSV")

Loaded 1470 rows, 42 columns from CSV


In [5]:
# Rename columns to match the SQL schems
df.columns = [camel_to_snake(c) for c in df.columns]
print("Renamed columns:", list(df.columns))

Renamed columns: ['age', 'attrition', 'business_travel', 'daily_rate', 'department', 'distance_from_home', 'education', 'education_field', 'employee_count', 'employee_number', 'environment_satisfaction', 'gender', 'hourly_rate', 'job_involvement', 'job_level', 'job_role', 'job_satisfaction', 'marital_status', 'monthly_income', 'monthly_rate', 'num_companies_worked', 'over_time', 'percent_salary_hike', 'performance_rating', 'relationship_satisfaction', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'work_life_balance', 'years_at_company', 'years_in_current_role', 'years_since_last_promotion', 'years_with_curr_manager', 'education_label', 'environment_satisfaction_label', 'job_involvement_label', 'job_satisfaction_label', 'relationship_satisfaction_label', 'work_life_balance_label', 'performance_rating_label', 'age_band', 'income_band']


In [6]:
print(DB_CONFIG)

{'user': 'postgres', 'password': 'dunith', 'host': 'localhost', 'port': '5432', 'database': 'hr_analytics'}


In [7]:
import sys
!{sys.executable} -m pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable


In [8]:
# Connect and Load
conn_str = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(conn_str)

In [9]:
print(df.columns.tolist())
print(df.shape)

['age', 'attrition', 'business_travel', 'daily_rate', 'department', 'distance_from_home', 'education', 'education_field', 'employee_count', 'employee_number', 'environment_satisfaction', 'gender', 'hourly_rate', 'job_involvement', 'job_level', 'job_role', 'job_satisfaction', 'marital_status', 'monthly_income', 'monthly_rate', 'num_companies_worked', 'over_time', 'percent_salary_hike', 'performance_rating', 'relationship_satisfaction', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'work_life_balance', 'years_at_company', 'years_in_current_role', 'years_since_last_promotion', 'years_with_curr_manager', 'education_label', 'environment_satisfaction_label', 'job_involvement_label', 'job_satisfaction_label', 'relationship_satisfaction_label', 'work_life_balance_label', 'performance_rating_label', 'age_band', 'income_band']
(1470, 42)


In [11]:
df = df.drop(columns=["employee_count"], errors="ignore")

df.to_sql(TABLE_NAME, engine, if_exists="append", index=False)
print(f"Loaded {df.shape[0]} rows into '{TABLE_NAME}' table")

Loaded 1470 rows into 'hr_employees' table


In [ ]:
# Verify
with engine.connect() as conn:
    result = conn.exec_driver_sql(f"SELECT COUNT(*) FROM {TABLE_NAME}")
    print("Row count in Postgres:", result.scaler())